# Phase 3d: counterparts in the reports, by targeted search and reading

Plan: `plans/phase_3d.md` with its amendments, including the independent manual check of the seven UBS target units that were class C (2026-09-23). Computation: `analysis/phase3d_terms.py` (search terms from the call sentences, fixed before searching), `analysis/phase3d_search.py` (search in all reports of the same bank up to the end of 2024), `analysis/phase3d_read.py` (reading result, every quote verified against the page); outputs in `data/phase3d/`. Descriptive and exploratory: no test, no register entry, no intervals.

The classifications were made by one reader, who also wrote the search terms. `counterparts_read.csv` starts with the seven units of the independent manual check; `checked_by_roman` is filled for them and empty for all others.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase3d"
pd.set_option("display.max_colwidth", 150)
d = pd.read_csv(OUT / "counterparts_read.csv", keep_default_na=False)
ORDER = ["2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4", "2023 annual report", "2024-Q1", "2024-Q2", "2024-Q3",
         "2024-Q4", "2024 annual report"]

## Search and integrity

In [ ]:
log = json.loads((OUT / "run_log.json").read_text())
terms = pd.read_csv(OUT / "search_terms.csv")
pd.Series({"units": len(d), "search terms, round 1": int((terms["round"] == 1).sum()),
           "search terms, round 2 (appended)": int((terms["round"] == 2).sum()),
           "round 1 unchanged since first search (sha256)": log["round1_sha256"][:16] + "...",
           "report files unchanged": "checked by hash in every run"}).to_frame("value")

Round 2 was used only for the seven JPMorgan units without any hit in round 1 (`outlook`, `guidance`, `net charge-off`); it found only accounting guidance. Window W1 covers the same-quarter report, all earlier reports including earlier annual reports, and for UBS the annual report of the same fiscal year; window ALL covers every report of the bank up to the end of 2024.

## Counts

In [ ]:
counts = pd.read_csv(OUT / "counts.csv", index_col=0)
counts.columns = pd.MultiIndex.from_tuples([("", "units"), ("", "not genuine")] +
                                           [("W1", c) for c in "ABC"] + [("ALL", c) for c in "ABC"])
counts

Of the 31 genuine UBS targets set after the acquisition, 16 have the same figure in a forward-looking sentence of the written record at the time of the call (W1), 13 have the same commitment with another or no figure, and 2 have no counterpart at all; over all reports up to the end of 2024 the split is 18, 11 and 2 (T27 and T16 moved from C to B under the consistency rule of 2026-09-24). The standalone savings programme (T3), announced before the acquisition, is counted separately and has no counterpart in the window. The cells for UBS guidance and other hold six and four genuine units, too few for shares.

JPMorgan's 19 genuine units have no A counterpart in either window. Guidance can by design not appear in the Exhibit 99.2 supplements, which contain no outlook; the same holds for the CET1 target and the First Republic targets of the event call, where only the preliminary bargain purchase gain has a counterpart (B).

## Not genuine commitments (excluded from the counts)

In [ ]:
d[d.genuine == "no"][["unit", "bank", "type", "origin", "call_sentence", "reason_not_genuine"]]

Nine of 70 units are not genuine commitments: five reported actuals or progress reports, one qualitative update of another unit's figure, one figure attributed to consensus, one reference to past rate cuts, and one thread that mixes metrics.

## Only the call: UBS targets without counterpart, checked independently

In [ ]:
d[d.checked_by_roman != ""][["unit", "origin", "figure_call", "class", "class_all", "pre_acquisition", "checked_by_roman"]]

After the independent manual check, four UBS targets set after the acquisition appear in no report at all: the NCL operational risk RWA of around 14 billion, the NCL credit and market risk RWA below 40 billion, the AT1 issuance plan and the HoldCo reduction to around 90 billion. The NCL capital release (T25) and the higher going-concern requirement (T32) move from C to B, because the 2023 annual report mentions both without a figure. The pre-acquisition standalone programme (T3) stays C, with its 2022 reports outside the window.

T27, B since the consistency rule of 2026-09-24 (kept as C until then): the 2023-Q4 report (p. 47) and the 2023 annual report (p. 193) state that “the run-down of positions in the Non-core and Legacy business division” is “expected to more than offset the effects of model updates and revised Basel III standards in 2024 and 2025”. The sentence is forward-looking about the same run-down but concerns Group RWA and gives no figure, so the NCL level below 40 billion remains call-only.

## The call first: counterpart only in a later report

In [ ]:
g = d[(d.genuine == "yes") & (d.first_report_all != "")].copy()
g["origin_q"] = g.origin.str[:7]
g["quarters_later"] = [ORDER.index(f) - ORDER.index(o) for f, o in zip(g.first_report_all, g.origin_q)]
g[g.quarters_later > 0][["unit", "type", "origin", "class", "class_all", "first_report_all", "quarters_later", "figure_call"]]

For six UBS units the first report with the same figure or commitment comes after the call. The clearest cases are the Basel III final estimate of about 5 % (call 2023-Q3, report 2024-Q2) and the effective tax rate of around 40 % (call 2023-Q4, report 2024-Q1); the CET1 guidance of around 14 % (call 2023-Q2, report 2023-Q4) follows the same pattern. In the other three cases (savings milestone for 2024, funding-cost savings, NCL below 5 %) the first report is the annual report, which is published after the call by construction.

## Consistency with phase 3b

In [ ]:
pd.DataFrame([
    ("C1 savings target 13bn (T24)", "same figure in the 2023-Q4 report", d.set_index("unit").loc["T24", "class"]),
    ("C2 integration total 13bn (T30)", "not in any report", d.set_index("unit").loc["T30", "class_all"]),
    ("C3 Basel III final 5% (T17)", "call first (2023-Q3), report 2024-Q2", d.set_index("unit").loc["T17", "first_report_all"]),
    ("C4 AT1 plan 2bn (T33)", "only in the call", d.set_index("unit").loc["T33", "class_all"]),
    ("C5 First Republic restructuring 2bn (T4)", "not restated in any supplement", d.set_index("unit").loc["T4", "class_all"])],
    columns=["case", "phase 3b reading", "phase 3d result"])

All five hand-read cases of phase 3b come out consistently: the integration total is B (the reports give quarterly costs, an indirect order of magnitude in 2023-Q2 and a revised estimate announced for 4Q23, but never the explicit total), the AT1 plan and the First Republic restructuring costs are C, and the Basel III figure reaches a report only in 2024-Q2.

## Reading

*Exploratory.* For UBS, about half of the genuine targets set after the acquisition are in the written record at the time of the call, and four of 31 are in no report up to the end of 2024; these four, like the two targets that appear only without figures, concern capital and the Non-core and Legacy run-down, the supervisory topics of the merger. For JPMorgan the comparison says little, because its supplements carry no forward-looking statements at all.

## Limits

One reader, who also chose the search terms; C means not found with the listed terms, not proven absent; the W1 window includes the annual report of the same fiscal year, which is published after the fourth-quarter call. `checked_by_roman` covers the seven units that were class C before the independent manual check. The text reconstruction of phase 3b joined some hyphenated words at line breaks ("Noncore" for "Non-core" in 18 of 752 cases). After the repair of 2026-09-23 the same search terms were run again on the repaired text: 168 new hit sentences in 19 units, none of them a forward-looking counterpart that would change a class.